In [1]:
import lsdb, torch
from lsdb_ml import CatalogDataLoader, CatalogDataset
import pandas as pd
from dask.distributed import Client
import time

# LSDB-ML Torch DataLoader

[LSDB-ML#22](https://github.com/lincc-frameworks/lsdb-ml/pull/22) implements the CatalogDataset & CatalogDataLoader classes, a Catalog-specific versions of the Dataset and DataLoader in Torch.

In Torch, the general purpose of the Dataset is to store and retreive individual data samples, whereas the DataLoader is meant to collect those samples into batches to be served via training.
For LSDB-ML, these responsibilities are adjusted slightly. The CatalogDataset itself provides batches of data, and so the CatalogDataLoader is a thinner wrapper of DataLoader that fixes some resulting kwargs into place (batch_size=None) and handles epoch management. 

>**Note**: The rationale for this difference is that the CatalogDataset leverages CatalogStreams, where access to a single row is much slower than just providing a batch. I could push on a solution that yields a single row but through some kind of stored batch infrastructure, if this difference is meaningfully disruptive to peoples usage. But I'm inclined to leave it as this, within the scope and priorities of Genesis Phase I (more bluntly, I don't think it's worth doing until we see that people actually use this and have issues with it).

## Toy Example

In [2]:
# generate a toy dataset
catalog = lsdb.generate_catalog(200, 10, seed=1)[["a", "b"]]
catalog

,a,b
npartitions=12,,
"Order: 0, Pixel: 0",double[pyarrow],double[pyarrow]
...,...,...
"Order: 0, Pixel: 10",...,...
"Order: 0, Pixel: 11",...,...


In [3]:
# Initialize lsdb-ml loaders
dataset = CatalogDataset(catalog, batch_size=32, seed=0)
loader = CatalogDataLoader(dataset)
dataset, loader

(<lsdb_ml.dataloader.CatalogDataset at 0x32ae00ce0>,
 <lsdb_ml.dataloader.CatalogDataLoader at 0x32ae4cc50>)

In [4]:
# Perform Training on a toy model
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
for epoch in range(2):
    print("-------")
    print(f"Epoch {epoch}")
    print("-------")
    for i, batch in enumerate(loader):
        print(f"Batch {i}")
        x, y = batch["a"].float()[:, None], batch["b"].float()[:, None]
        loss = torch.nn.functional.mse_loss(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

-------
Epoch 0
-------
Batch 0
Batch 1
Batch 2
Batch 3
Batch 4
Batch 5
Batch 6
-------
Epoch 1
-------
Batch 0
Batch 1
Batch 2
Batch 3
Batch 4
Batch 5
Batch 6


In [5]:
# Example of prediction
model.eval()
ids, preds = [], []
with torch.no_grad():
    for batch in loader:
        x, y = batch["a"].float()[:, None], batch["b"].float()[:, None]
        preds.append(model(x).squeeze(1))

result = pd.DataFrame({"pred": torch.cat(preds).numpy()})
result

,pred
0,-0.068031
1,0.116497
...,...
198,-0.152767
199,0.005972


### A generic torch DataLoader works as well

In [13]:
# As long as we set batch_size=None
from torch.utils.data import DataLoader
torch_loader = DataLoader(dataset, batch_size=None)

model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
for epoch in range(2):
    print("-------")
    print(f"Epoch {epoch}")
    print("-------")
    for i, batch in enumerate(torch_loader):
        print(f"Batch {i}")
        x, y = batch["a"].float()[:, None], batch["b"].float()[:, None]
        loss = torch.nn.functional.mse_loss(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

-------
Epoch 0
-------
Batch 0
Batch 1
Batch 2
Batch 3
Batch 4
Batch 5
Batch 6
-------
Epoch 1
-------
Batch 0
Batch 1
Batch 2
Batch 3
Batch 4
Batch 5
Batch 6


## A "Real" Example

In [6]:
cone = lsdb.ConeSearch(ra=300.0, dec=0.0, radius_arcsec=6000)

ztf_dr24 = lsdb.open_catalog('s3://ipac-irsa-ztf/ztf/enhanced/dr24/lc/hats', search_filter=cone)
ztf_dr24

,objectid,filterid,objra,objdec,lightcurve
npartitions=15,,,,,
"Order: 5, Pixel: 7521",int64[pyarrow],int8[pyarrow],float[pyarrow],float[pyarrow],"nested<hmjd: [double], mag: [float], magerr: [..."
...,...,...,...,...,...
"Order: 6, Pixel: 30129",...,...,...,...,...
"Order: 6, Pixel: 30132",...,...,...,...,...


In [7]:
few_ztf = ztf_dr24.head(10)
few_ztf

Computing Catalog:   0%|          | 0/2 [00:00<?, ?it/s]

objectid  filterid       objra    objdec  \
_healpix_29                                                             
2117250230809636520  1483101100009303         1  299.372681 -1.374983   
2117250230809750491   438110100013821         1  299.372681 -1.374965   
2117250230810071224   438210100020102         2   299.37265  -1.37497   
2117250231476284600   438210100020025         2  299.369904 -1.372971   
2117250231476694359  1483201100013192         2  299.369843  -1.37297   
2117250233223126243  1483201100064253         2  299.370483 -1.371176   
2117250233223805941   438210100060703         2  299.370422 -1.371107   
2117250233632727148  1483101100027387         1   299.36792 -1.370762   
2117250233632840321  1483201100013098         2  299.368011 -1.370741   
2117250233633030543   438210100038717         2  299.368011 -1.370711   

                                                            lightcurve  
_healpix_29                                                             
2117250230809636520  [{hmjd: 59060.34726, mag: 16.610256, magerr: 0...  
2117250230809750491  [{hmjd: 58228.48188, mag: 16.607037, magerr: 0...  
2117250230810071224  [{hmjd: 58228.45094, mag: 15.94887, magerr: 0....  
2117250231476284600  [{hmjd: 58360.27693, mag: 21.009789, magerr: 0...  
2117250231476694359  [{hmjd: 59368.44171, mag: 21.312296, magerr: 0...  
2117250233223126243  [{hmjd: 59761.37488, mag: 21.853188, magerr: 0...  
2117250233223805941  [{hmjd: 58323.25714, mag: 20.782326, magerr: 0...  
2117250233632727148  [{hmjd: 59750.43672, mag: 21.204863, magerr: 0...  
2117250233632840321  [{hmjd: 59061.43499, mag: 20.122601, magerr: 0...  
2117250233633030543  [{hmjd: 58252.48433, mag: 20.262228, magerr: 0...

In [8]:
ztf_dr24["lightcurve"].columns

['hmjd', 'mag', 'magerr', 'clrcoeff', 'catflags']

In [9]:
import torch.nn as nn
 
MAG_IDX = 2    # feature column holding the normalized magnitude
FLAG_IDX = 4   # feature column holding the "this epoch is masked" flag
 
 
class LightCurveEncoder(nn.Module):
    def __init__(self, in_dim=5, hidden=64, emb_dim=32, num_layers=2, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(in_dim, hidden)
        self.gru = nn.GRU(hidden, hidden, num_layers=num_layers, batch_first=True,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.to_emb = nn.Linear(2 * hidden, emb_dim)
 
    def forward(self, x, pad):
        h, _ = self.gru(torch.relu(self.inp(x)))                 # (B, L, 2H)
        valid = (~pad).unsqueeze(-1).to(h.dtype)
        pooled = (h * valid).sum(1) / valid.sum(1).clamp(min=1)  # masked mean pool
        return h, self.to_emb(pooled)                            # per-step, per-curve
 
 
class MaskedLightCurveModel(nn.Module):
    """forward(x, pad) -> (recon, emb); recon is (B, L), emb is (B, emb_dim)."""
 
    def __init__(self, in_dim=5, hidden=64, emb_dim=32, num_layers=2, dropout=0.1):
        super().__init__()
        self.encoder = LightCurveEncoder(in_dim, hidden, emb_dim, num_layers, dropout)
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, 1))
 
    def forward(self, x, pad):
        h, emb = self.encoder(x, pad)
        return self.head(h).squeeze(-1), emb
 
    @staticmethod
    def random_mask(x, pad, frac=0.15):
        """Hide a random fraction of valid mags. Returns (x_masked, mask, target)."""
        target = x[..., MAG_IDX].clone()
        mask = (torch.rand_like(target) < frac) & ~pad
        x = x.clone()
        x[..., MAG_IDX] = x[..., MAG_IDX].masked_fill(mask, 0.0)
        x[..., FLAG_IDX] = x[..., FLAG_IDX].masked_fill(mask, 1.0)
        return x, mask, target
 
    @staticmethod
    def loss(recon, target, mask):
        """Huber loss on masked epochs only."""
        return nn.functional.huber_loss(recon[mask], target[mask])

def masked_mean(v, valid):
    return (v * valid).sum(1, keepdim=True) / valid.sum(1, keepdim=True).clamp(min=1)

def make_inputs(batch, lc_col="lightcurve"):
    lc = batch[lc_col]
    pad = ~lc["mask"]                                   # our mask is True on real epochs
    valid = lc["mask"].double()
    # Sort each curve by time; padding sorts to the end because it is set to +inf.
    order = lc["hmjd"].masked_fill(pad, float("inf")).argsort(dim=1)
    take = lambda v: v.gather(1, order)
    hmjd, mag = take(lc["hmjd"]), take(lc["mag"])
    t = (hmjd - masked_mean(hmjd, valid)) / 365.0       # in float64, before casting: hmjd ~ 6e4
    mag = mag - masked_mean(mag, valid)                 # per-curve centered magnitude
    bad = (take(lc["catflags"]) != 0).double()          # catflags is a bitmask; use "any flag set"
    flag = torch.zeros_like(mag)                        # set by random_mask
    x = torch.stack([t, mag, take(lc["magerr"]), take(lc["clrcoeff"]), bad, flag], dim=-1)
    return x.masked_fill(pad.unsqueeze(-1), 0.0).float(), pad

In [10]:
time0 = time.time()
with Client(n_workers=2, threads_per_worker=1, memory_limit="14GB") as client:
    print(client.dashboard_link)
    ztf_ds = CatalogDataset(
        ztf_dr24[["objectid", "lightcurve"]],   # only what you need
        batch_size=64, seed=0, client=client,
        partitions_per_chunk=2,         # keep both dask workers busy
        max_length={"lightcurve": 64},         # fixed shapes; truncates longer curves
    )
    ztf_loader = CatalogDataLoader(ztf_ds)

    model = MaskedLightCurveModel(in_dim=6)
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
    for epoch in range(1):
        for i, batch in enumerate(ztf_loader):
            if i == 0:
                time1=time.time()
                print(f"Time to First Batch: {time1-time0} s")
            if i%1000==0:
                print(f"Batch {i}")
            x, pad = make_inputs(batch)
            x_masked, m, target = model.random_mask(x, pad)
            recon, emb = model(x_masked, pad)
            loss = model.loss(recon, target, m)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

http://127.0.0.1:8787/status
Time to First Batch: 209.06105208396912 s
Batch 0
Batch 1000
Batch 2000
Batch 3000
Batch 4000
Batch 5000
Batch 6000
Batch 7000
Batch 8000
Batch 9000
Batch 10000
Batch 11000
Batch 12000
Batch 13000
Batch 14000
Batch 15000
Batch 16000
Batch 17000
Batch 18000
Batch 19000
Batch 20000
Batch 21000
Batch 22000
Batch 23000
Batch 24000
Batch 25000
Batch 26000
Batch 27000
Batch 28000
Batch 29000
Batch 30000
Batch 31000
Batch 32000
Batch 33000
Batch 34000
Batch 35000
Batch 36000


In [11]:
# 4 mins for ppc=4, 1 24GB worker
# 2 (3?) mins for ppc=2, 2 12GB workers

## TODO: See how GPUs go